## Apresentação ✒️

Notebook destinado a demonstrar a primeira análise que pode ser realizada para a qualidade de respostas dos modelos generativos de linguagem, conhecidas como LLM's, empregados em cenários como chatbots. Durante o desenvolvimento de tais aplicações, enquadrando-as dentro de um contexto de ciência de dados, os modelos generativos podem ser compreendidos como os modelos de machine learning (ML) utilizados para a resolução de um problema passível de predição, mediante a inserção de dados. 

Uma vez que possui predição, realiza resposta em função de certos inputs - que nesse contexto - passam a ser as mensagens dos usuários, que em conjunto com os parâmetros do modelo produzem determinados outputs, os quais podem ser comparados com uma `ground truth` considerada, para avaliar a qualidade de resposta do modelo. Da mesma forma, nesse cenário, diferente de outros como em regressão ou classificação, a resposta de tais modelos não são sujeitos a um intervalo ou correspondem a um valor de erro médio, mas são textos, os quais, para a sua análise, demandam necessariamente de uma análise qualitativa.

Contudo, como desempenhá-la de uma forma mais automatizada, de modo em que a dependência dos curadores humanos - que atuam como provedores da ground truth e avaliadores canônicos da resposta do modelo - diminua e também quantificar aquilo que é essencialmente qualitativo? Para responder a essa pergunta, o presente notebook irá apresentar um primeiro sistema de avaliação de tais modelos generativos, considerando o uso de algumas bibliotecas, que avaliam a qualidade da resposta segundo a sua perspectiva vetorial - por `cossine similarity` -, bem como o método conhecido como `LLM as a Judge`. As primeiras bibliotecas serão as bibliotecas do BERT Score e a Sentence Embedding, as quais analisam a proximidade dos vetores de cada termo presente numa mensagem e a sentença como um todo, respectivamente. 

Para a avaliação das respostas será utilizado um `foundation agent` num contexto de conversational RAG, que irá responder a determinadas perguntas em função de uma base de conhecimento provisionada. A avaliação irá considerar as seguintes colunas do dataset : 

- user message (pergunta)
- kb (base de conhecimento do modelo utilizada em sua resposta)
- ground truth kb (o Kb que seria esperado pelo modelo retornar para a sua resposta)
- ground truth response (a resposta esperada que o modelo deveria provisionar)

A avaliação terá duas camadas de consideração : A primeira irá avaliar a qualidade do modelo com base em sua resposta e comparação com a ground truth. A segunda também irá considerar a precisão do mecanismo de busca dos Kb's, dado a necessidade de avaliar a qualidade desse durante o contexto de conv RAG, dado que esses podem contribuir com a boa experiência do usuário em função de boas respostas ou não - garbage in, garbage out. No entanto, pode-se assumir de maneira proxy a qualidade do mecanismo de busca com base nas respostas geradas do modelo em relação à ground truth, devido a natureza de dependência inerente a tal aplicação. 

**Legenda :** 

- Ground truth: Refere-se a verdade canônica que assumi-se que o modelo deveria responder. É como se fosse o gabarito de uma prova, que demarca a resposta correta.
- Foundation Agent: Nome dado aos agentes conversacionais baseados nos modelos de linguagem. Recebe esse nome para fins de segmentação perante aos outros tipos de agents já presentes na literatura. 

**Pontos de atenção :** 

A abordagem conhecida como LLM as a Judge não é a única forma de avaliar as respostas dos modelos generativos, havendo também frameworks como RAGAS, que também consideram o aspecto qualitativo da resposta do modelo e a quantificam. Com o judge, tal processo também ocorre, dando-se a partir de rúbricas (valor), junto de uma justificativa do motivo pelo qual o modelo realizou tal pontuação. De forma geral, as rúbricas variam de intervalo, podendo ser de 0 a 1, de 1 a 5 ou booleana, isto é, de 1 para certo e 0 para errado. O mais importante para tais valores é fomentar um cenário de análise quantitativa, mas sobretude de formação de uma curva de distribuição que possibilite, posteriormente, a realização de testes de hipótese com base na consideração dos curadores. Os tipos de teste de hipótese variam, sendo comumente adotado o `z-score` ou o `test-t`.

### Library 📓

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm

from typing import Dict, List

from IPython.display import Markdown

from scipy.stats import ttest_rel


from features.clean_memory import CleanMemory

from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

from prompts.system_message import system_message
from prompts.check_context import check_context_prompt
from prompts.contextualize_message import contextualize_prompt

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Inicializando o modelo de LLM

In [ ]:
# API reference : your-api-key

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [4]:
llama_2 = "llama3-70b-8192"
qwen_qwen = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama = "llama-3.3-70b-versatile"
deepseek = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Cossine Similarity Lib

Método responsável por promover uma análise objetiva - e quantitativa - da qualidade da resposta do modelo. Isso é feito segundo a compreensão que os termos presentes em linguagem natural podem ser compreendidos como vetores presentes num espaço dimensional no qual podem ser posicionados e mensurados a distância entre si, na forma em que quanto mais próximos um dos outros vetorialmente, tem-se que - tudo o mais constante - estarão também semanticamente. 

In [ ]:
def sentence_embedding_similarity(
        dataset: DataFrame,
        column_ground_truth: str, 
        column_response_model: str, 
        index: int
    ) -> str:
    """ 
    Computes the cosine similarity between embeddings of 'ground truth' and 'response model' 
    from a specified row in the DataFrame.

    This function uses a pre-trained SentenceTransformer model to generate embeddings 
    for the specified 'response' and 'ground truth' texts in the DataFrame. The embeddings are 
    compared using cosine similarity to measure their semantic similarity.

    Args:
        dataset (DataFrame): A pandas DataFrame containing 'prompt' and 'response' columns.
        column_ground_truth: The name of grond truth's column.
        column_response_model: The name of response model's column.
        index (int): The row index in the DataFrame from which to extract the texts.

    Returns:
        float: The cosine similarity score between the embeddings of 'ground truth' and 'response model'.
    """
    model = SentenceTransformer("all-MiniLM-L6-v2")

    ground_truth = dataset.iloc[index][column_ground_truth]
    response_model = dataset.iloc[index][column_response_model]
    
    ground_truth_embedding = model.encode(ground_truth)
    response_embedding = model.encode(response_model)

    # Como a biblioteca SentenceTransformers espera vetores em 2D, 
    # tive que adicionar mais uma dimensão a cada embedding, formando
    # os respectivos expand embeddings a seguir, tanto para o prompt
    # quanto para a resposta gerada. 

    expand_prompt_embedding = np.expand_dims(ground_truth_embedding, axis=0)
    expand_response_embedding = np.expand_dims(response_embedding, axis=0)

    cossine_similarity = pairwise_cos_sim(
        expand_prompt_embedding, 
        expand_response_embedding
    )

    cossine_similarity_value = round(cossine_similarity[0].item(), 1)
    return cossine_similarity_value

### Dataset utilizado

In [6]:
""" 
Para esse run, estou carregando o dataset que já foi avaliado pelo judge. 
Para obter a própria avaliação, basta carregar o dataset `dataset_response_model`, 
o qual contém a resposta do modelo, junto do Kb recuperado. 
"""

file_name = "short_dataset"

df = pd.read_excel(f"./data/{file_name}.xlsx", engine="openpyxl")

In [7]:
df = df.drop("Unnamed: 0", axis=1)

In [8]:
df

,Question,Original Response,Response Adv Message - 1,Response Adv Message - 2,Kb Recovered Adv - 1,Kb Recovered Adv - 2
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,"Não, isso não é verdade. Os ghouls precisam se...","Não, isso não é verdade. Os ghouls precisam se...","['a como uma extensão do seu corpo, permitindo...","['idas, vivendo entre duas naturezas conflitan..."
1,Como e por que foi criada a organização CCG?,Os cenários noturnos em Tokyo Ghoul contribuem...,"Não, isso não é verdade. Os cenários noturnos ...","Não, isso não é verdade. Os cenários noturnos ...",['A ambientação principal de Tokyo Ghoul é a p...,['A ambientação principal de Tokyo Ghoul é a p...
2,O que acontece com Ken Kaneki após o transplan...,Os elementos de horror corporal (body horror) ...,"Não, isso não é verdade. Os ghouls em Tokyo Gh...","Desculpas, não sei responder sobre isso no mom...",['inhas entre \nbem e mal tornam-se tênues. \n...,['inhas entre \nbem e mal tornam-se tênues. \n...
3,Quais diferentes visões de convivência entre g...,"Kaneki, como um meio-ghoul, personifica a luta...","Não, isso não é verdade. Kaneki, como um meio-...","Não, isso não é verdade. Kaneki, como um meio-...",['mo e ao preconceito estrutural. Esse subtext...,"['ls, utilizando quinques — armas feitas a par..."
4,: Qual é o significado de “One-Eyed King” no u...,A ambientação urbana de Tokyo Ghoul reflete o ...,"Desculpas, não sei responder sobre isso no mom...","Desculpas, não sei responder sobre isso no mom...",['inhas entre \nbem e mal tornam-se tênues. \n...,['mo e ao preconceito estrutural. Esse subtext...


In [9]:
df.shape

(5, 6)

#### Verificando sua integridade

In [10]:
df.isnull().sum()

Question                    0
Original Response           0
Response Adv Message - 1    0
Response Adv Message - 2    0
Kb Recovered Adv - 1        0
Kb Recovered Adv - 2        0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.shape

(29, 15)

### Avaliação - BERT Score

In [12]:
model = "distilbert-base-uncased"

In [15]:
# Interagindo com o modelo. 

bert_score = evaluate.load("bertscore")

bert_score.compute(
    predictions = [df.loc[0, "Response Adv Message - 1"]],
    references  = [df.loc[0, "Original Response"]], 
    model_type  = model  
)

{'precision': [0.8608095049858093],
 'recall': [0.8555428385734558],
 'f1': [0.8581681251525879],
 'hashcode': 'distilbert-base-uncased_L5_no-idf_version=0.3.12(hug_trans=4.50.3)'}

In [15]:
# Escolhendo apenas a média harmônica. 

bert_score_eval = bert_score.compute(
    predictions = [df.loc[5, "Kbs Recovered"]],
    references  = [df.loc[5, "Base de conhecimento"]], 
    model_type  = model  
)["f1"]

f1_score = round(bert_score_eval[0], 3)
print(f"F1-score: {f1_score}")

F1-score: 0.802


#### Iterando com o modelo sobre o dataset

Iteração responsável pela formação da métrica que considera a corretude da resposta e da base de conhecimento recuperada, em função da base de conhecimento recuperada. Para reiteirar, concebe-se que quanto mais próximo de 1 for o valor encontrado, mais próximo vetorialmente está os termos e, portanto, são semanticamente mais próximos. 

In [11]:
f1_score_response = []
f1_score_kb_recovered = []

In [12]:
kb_column = "Kbs Recovered"
canon_kb_column = "Base de conhecimento"

response_column = "Response Model"
ground_truth_column = "Ground Truth"

In [16]:
%%time

for _, row in tqdm(df.iterrows(), desc="Avaliando com BERT Score:", total=df.shape[0]):

    bert_score_eval = bert_score.compute(
        predictions = [ row[f"{response_column}"] ],
        references  = [ row[f"{ground_truth_column}"] ],
        model_type  = model
    )["f1"]

    f1_score = round(bert_score_eval[0], 1)
    f1_score_response.append(f1_score)
    # f1_score_kb_recovered.append(f1_score)


Avaliando com BERT Score:: 100%|██████████| 29/29 [00:08<00:00,  3.29it/s]

CPU times: total: 30.8 s
Wall time: 8.83 s


In [17]:
# Adicionando as métricas geradas ao dataset, lembrando que o 
# valor encontrado se refere a média harmônica. 

df["bert_score_response"] = f1_score_response
df["bert_score_kb"] = f1_score_kb_recovered

### Avaliação - Cossine Similarity 

In [ ]:
cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_ground_truth   = "Ground Truth",
    column_response_model = "Response Model", 
    index                 = 5
)

print(f"Cossine Similarity: {cossine_similarity}")

Cossine Similarity: 0.556


In [19]:
cossine_similarity_response = []
cossine_similarity_kb = []

In [22]:
%%time

for i in tqdm(range(29), desc="Avaliando com C. Smilarity:"):

    cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_ground_truth   = ground_truth_column,
    column_response_model = response_column, 
    index                 = i
    )

    cossine_similarity_response.append(cossine_similarity)
    # cossine_similarity_kb.append(cossine_similarity)

Avaliando com C. Smilarity:: 100%|██████████| 29/29 [00:45<00:00,  1.56s/it]

CPU times: total: 29.9 s
Wall time: 45.3 s


In [23]:
df["cossine similarity response"] = cossine_similarity_response
df["cossine similarity kb"] = cossine_similarity_kb

### Avaliação - LLM as a Judge

#### Prompt

In [24]:
judge_prompt ="""\
        <role>
        Aja como um especialista em avaliação de chatbots conversacionais, em especial 
        para aplicações no contexto de Conversational RAG, no qual as respostas geradas
        pelo modelo generativo são baseadas em uma base de conhecimento. 
        A sua tarefa é avaliar a qualidade da resposta gerada pelo modelo e da
        base de conhecimento recuperada pelo algoritmo do RAG.
        </role>

        <metricas>
        Essas são as métricas com as quais você deve considerar para a análise da 
        resposta do modelo generativo e da base de conhecimento recuperada. 
        <corretude>: Avalia o quão correta está a <response_model> com base na <ground_truth>.
        <completude>: Avalia o quão completa está a <response_model> em relação ao <kb>.
        <relevancia>: Avalia o quão resolutiva foi a <response_model> em relação à <user_message>.
        <resposta_embasada>: Avalia o quão embasada está a <response_model> em relação <canon_kb>
        <embasamento>: Avalia o quão o <kb> está ressonante ao <canon_kb>.
        </metricas>

        <exemplos>
        Exemplos que ilustram como você deve avaliar a resposta do modelo generativo. 
        <exemplo1> **Pergunta**: `Quem é o One-Eyed King em Tokyo Ghoul e qual é o seu papel na história?` **Resposta do modelo**: `O One-Eyed King é uma figura simbólica e messiânica no universo de Tokyo Ghoul. Inicialmente envolto em mistério, esse título representa a esperança de que ghouls e humanos possam coexistir. A identidade do One-Eyed King é revelada como sendo Ken Kaneki, que assume esse papel com o objetivo de unir facções opostas e pôr fim ao ciclo de violência. Sua posição desafia tanto os ghouls radicais quanto o CCG, tornando-o uma peça-chave na transformação do equilíbrio de poder.` **Avaliação**: Corretude: A resposta fornecida está de acordo com a ground truth - 1.0. Completude: A resposta cobre completamente a base de conhecimento fornecida. -1. Relevância: A resposta do modelo é perfeitamente relevante ao responder a pergunta, estando correta e completa - 1. Resposta embasada: A resposta do modelo está aderente à base de conhecimento canônica - 1.<exemplo1>
        <exemplo2> **Pergunta**: `Qual é o papel do CCG em Tokyo Ghoul?` **Resposta do modelo**: `O CCG é uma instituição criada para manter a paz entre humanos e ghouls, promovendo encontros diplomáticos com representantes das duas espécies. Eles incentivam acordos de paz, mas enfrentam resistência por parte de ghouls extremistas como o Aogiri Tree.` **Avaliação**: Corretude: Há distorções com a ground truth, pois o CCG não promove acordos de paz, mas combate ativo aos ghouls. - 0.4 Completude: A resposta ignora aspectos como os Investigadores, as armas quinques e a função militarizada, não estando completa quando comparada à ground truth. - 0.4. Relevância: Apesar dos erros evidenciados, a resposta do modelo foca-se em responder ao tema CCG. - 0.8. Resposta embasada: A resposta do modelo está parcialmente aderente à base de conhecimento canônica - 0.8 <exemplo2>
        <exemplo3> **Pergunta**: `Quem é Yoshimura e como ele se relaciona com os outros personagens?` **Resposta do modelo**: `Yoshimura é um cientista que trabalha para o CCG e foi responsável por criar as RC cells. Ele também desenvolveu uma vacina contra os ghouls e participou de várias guerras contra criaturas chamadas titãs.` **Avaliação**: Corretude:  Fatos totalmente errados (mistura elementos de outras séries como Attack on Titan), não estando adequada à ground truth - 0.0. Completude:  Nenhuma informação correta ou pertinente. - 0.0. Relevância: A resposta é completamente deslocada do universo de Tokyo Ghoul, não respondendo à pergunta. - 0.0. Resposta embasada: A resposta do modelo não está embasada na base de conhecimento, promovendo um erro na resposta do modelo - 0.0<exemplo3>
        </exemplos>

        <instrucao>
        **Somente** responda em português. 
        Avalie a <resposta_do_modelo>, considerando à <ground_truth> e <pergunta> fornecida, com base em <metricas>.
        A sua avaliação conta com um conjunto de rúbrica (valor) e justificativa. A rúbrica varia segundo um intervalo de 0 a 1, sendo 1 a máxima pontuação e 0 a mínima. Pontuação de 0.8 indica acerto parcial, com falha em pelo menos uma das métricas e 0.4 um erro parcial, com pelo menos um acerto nas métricas. 
        Além da pontuação, forneça uma justificativa que fundamentou a sua pontuação, com uma explicação detalhada acerca dela, estrutura em formato de premissa e conclusão.
        Para a sua avaliação, considere as métricas fornecidas em <metricas> e avalie a resposta do modelo para cada uma das métricas, conforme demonstrado em <exemplos>.
        </instrucao>

        <variaveis>
        <user_message>: {user_message}
        <response_model>: {response_model}
        <canon_kb>: {canon_kb}
        <kb_recovered>: {kb_recovered}
        </variaveis>

        <resposta>
        A sua resposta deve considerar as métricas de <corretude>, <completude> e <relevancia>.
        Formate a sua resposta utilizando o seguinte template: {format_instructions}
        </resposta>
    """

In [25]:
""" 
Criando a formatação da resposta esperada pelo judge. 
"""

class JudgeEval(BaseModel):
    criterio: str = Field(description="Critério avaliado")
    rubrica: float = Field(description="Valor da métrica avaliada")
    justificativa: str = Field(description="Justificativa da avaliação")

class JudgeOutput(BaseModel):
    avaliacoes: List[JudgeEval]

parser = PydanticOutputParser(pydantic_object=JudgeOutput)

In [ ]:
def judge(
        user_message: str, 
        response_model: str, 
        ground_truth: str,
        canon_kb: str,
        kb_recovered: str,  
        llm = llm, 
        parser = parser
    ) -> str:
    """
    Evaluates the quality of a generative model's response using a language model (LLM) and predefined criteria.

    This function builds a prompt based on a question, the model's response, and the ground truth answer. It uses
    a chain-of-thought evaluation strategy with a language model to assess the response against four criteria:
    correctness, completeness, relevance, and overall performance.

    Args:
        question (str): The original user question that was asked.
        response_model (str): The response generated by the model being evaluated.
        ground_truth (str): The reference answer considered to be correct.
        llm: The language model used for generating the evaluation (default: global `llm`).
        parser: The parser used to structure and validate the LLM output (default: global `parser`).

    Returns:
        dict: A dictionary containing the evaluation results with metrics including rubric (score) and justification
              for each criterion: correctness, completeness, relevance, and overall performance.
    """ 
    judge_prompt_template = PromptTemplate(
        template          = judge_prompt, 
        input_variables   = ["response_model", 
                             "user_message", 
                             "ground_truth", 
                             "canon_kb", 
                             "kb_recovered"],
        partial_variables = {"format_instructions": parser.get_format_instructions()}
    )

    judge_chain = judge_prompt_template | llm | JsonOutputParser()

    try:
        judge_response = judge_chain.invoke(
            {
                "user_message": user_message, 
                "response_model": response_model,
                "ground_truth": ground_truth, 
                "canon_kb": canon_kb,
                "kb_recovered": kb_recovered
            }
        )
        return judge_response
    except Exception as e:
        print(f"[ERRO] Falha ao avaliar a linha com question: '{user_message[:30]}...'. Detalhes: {e}")
        return None

### Testando o Judge

In [20]:
canon_kb = df["Base de conhecimento"][5]
kb_recovered = df["Kbs Recovered"][5]
user_message = df["Question"][5]
ground_truth = df["Ground Truth"][5]
response_model = df["Response Model"][5]

In [ ]:
print("Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:")

print(f"""
Pergunta: {user_message}\n\n
Ground Truth: {ground_truth}\n\n 
Response Model: {response_model}\n\n 
Canon Kb: {canon_kb}\n\n
Kb recuperado: {kb_recovered}
""")

In [ ]:
    %%time

    """
    Testando o judge formato para um conjunto de texto 
    abitrariamente escolhidos. 
    """

    judge_response = judge(
        user_message   = user_message, 
        response_model = response_model, 
        ground_truth   = ground_truth, 
        canon_kb       = canon_kb, 
        kb_recovered   = kb_recovered
    )

CPU times: total: 125 ms
Wall time: 3.46 s


In [23]:
judge_response["avaliacoes"]

[{'criterio': 'Corretude',
  'rubrica': 0.8,
  'justificativa': 'A resposta do modelo está de acordo com a ground truth em relação à evolução de Ken Kaneki e sua interação com diferentes facções ao longo da história de Tokyo Ghoul. No entanto, há alguns detalhes que não estão presentes na resposta, como a menção à sua condição de meio-ghoul e sua busca por sobrevivência e aceitação.'},
 {'criterio': 'Completude',
  'rubrica': 0.6,
  'justificativa': 'A resposta do modelo cobre a maior parte da base de conhecimento fornecida, mas falta mencionar alguns detalhes importantes, como a ambientação do Anteiku e a sua importância como um local de encontro para personagens-chave.'},
 {'criterio': 'Relevancia',
  'rubrica': 0.9,
  'justificativa': 'A resposta do modelo é muito relevante à pergunta, pois descreve a evolução de Ken Kaneki e sua interação com diferentes facções ao longo da história de Tokyo Ghoul. No entanto, há alguns detalhes que não estão diretamente relacionados à pergunta, com

In [24]:
metric_values = []

for i in judge_response["avaliacoes"]:
    metric_values.append(i["rubrica"])

# Média harmônica que informa a qualidade global da resposta
# do modelo generativo. 
global_performance = round(len(metric_values) / (1/metric_values[0] + 1/metric_values[1] + 1/metric_values[2] + 1/metric_values[3] + 1/metric_values[4]), 2)
global_performance

0.75

### Iterando com as informações para o Judge

In [25]:
# Criando as colunas no dataset para cada uma das métricas.

for criterio in ['corretude', 'completude', 'relevancia', 'resposta embasada', 'embasamento']:
    df[f'{criterio}_rubrica'] = None
    df[f'{criterio}_justificativa'] = None

In [26]:
for i in tqdm(range(30), desc="Gerando a avaliação"):
    
    user_message = df["Question"][i]
    ground_truth = df["Ground Truth"][i]
    response_model = df["Response Model"][i] 
    kb_recovered = df["Kbs Recovered"][i]
    canon_kb = df["Base de conhecimento"][i]

    # Chama o judge
    result = judge(
    user_message   = user_message, 
    response_model = response_model, 
    ground_truth   = ground_truth, 
    canon_kb       = canon_kb, 
    kb_recovered   = kb_recovered
    )

    # Lista de avaliações
    if result is None:
        continue 
    
    metric_values = []
    avaliacoes = result["avaliacoes"]  

    for avaliacao in avaliacoes:
        criterio = avaliacao["criterio"].lower()  
        rubrica = avaliacao["rubrica"]
        justificativa = avaliacao["justificativa"]

        df.at[i, f"{criterio}_rubrica"] = rubrica
        df.at[i, f"{criterio}_justificativa"] = justificativa

        metric_values.append(rubrica)

Gerando a avaliação: 100%|██████████| 30/30 [13:54<00:00, 27.82s/it]


In [ ]:
# Para salvar o dataset após a avaliação do judge. 
# Recomendado por conta de finalização do Kernal do Jupyter. 

file_name = "dataset_with_judge"
df.to_excel(f"{file_name}.xlsx")

### Dataset final

In [ ]:
""" 
Ponto de atenção : 
A decisão de deixar as métricas BERT Score e Cossine Similarity com apenas uma casa
decimal se relaciona ao fato de melhor conseguir fazer investigações posteriores, 
como matriz de correlação, curvas de distribuição e visualização gráfica, para 
conseguir ver - nos primeiros casos - se as métricas apresentam similar distribuição - 
e, no segundo, como para a determinada amostra o modelo está desempenhando. 
"""

df

,Question,Ground Truth,Base de conhecimento,Response Model,Kbs Recovered,corretude_rubrica,corretude_justificativa,completude_rubrica,completude_justificativa,relevancia_rubrica,relevancia_justificativa,resposta embasada_rubrica,resposta embasada_justificativa,embasamento_rubrica,embasamento_justificativa,bert_score_response,bert_score_kb,cossine similarity response,cossine similarity kb
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,Os ghouls são seres são fisicamente muito seme...,Os ghouls são criaturas muito semelhantes aos ...,['inhas entre \nbem e mal tornam-se tênues. \n...,0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo apresenta uma boa compree...,1.0,A resposta do modelo é perfeitamente relevante...,0.8,A resposta do modelo está embasada na base de ...,0.4,A base de conhecimento recuperada pelo algorit...,1.0,0.8,1.0,0.6
1,Como e por que foi criada a organização CCG?,À medida que cresciam os conflitos e as mortes...,"Originalmente, a sociedade humana desconhece a...",A organização CCG (Comissão de Contra-Ghoul) f...,['ca por um meio-\ntermo entre a sobrevivência...,0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo cobre a criação do CCG e ...,0.9,A resposta do modelo é muito relevante à pergu...,0.8,A resposta do modelo está embasada na base de ...,0.7,O conhecimento recuperado pelo algoritmo do RA...,0.9,0.9,0.8,0.6
3,Quais diferentes visões de convivência entre g...,Existem facções que defendem a paz e a coexist...,Alguns grupos de ghouls defendem a paz e tenta...,"Em Tokyo Ghoul, existem diferentes visões de c...",['entre humanos e ghouls. Alguns \ngrupos de g...,0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo apresenta uma visão geral...,0.9,A resposta do modelo está perfeitamente releva...,0.8,A resposta do modelo está embasada na base de ...,0.8,O conhecimento recuperado pelo algoritmo do RA...,0.9,0.9,0.7,0.8
4,: Qual é o significado de “One-Eyed King” no u...,O “One-Eyed King” (Rei de Olho Único) é uma fi...,Espécime de figura messiânica para alguns ghou...,"No universo de Tokyo Ghoul, o ""One-Eyed King"" ...",['inhas entre \nbem e mal tornam-se tênues. \n...,0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo apresenta uma visão geral...,1.0,A resposta do modelo é perfeitamente relevante...,0.8,A resposta do modelo está embasada na base de ...,0.8,A base de conhecimento recuperada pelo algorit...,0.9,0.8,0.9,0.8
5,De que forma Ken Kaneki se envolve com diferen...,"Após se tornar meio-ghoul, Kaneki entra em con...","Após se tornar meio ghoul, Kaneki encontra ref...",Ken Kaneki se envolve com diferentes facções a...,"['nas do mangá, \ninfluenciando tendências est...",0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo cobre a maior parte da ba...,0.9,A resposta do modelo é muito relevante à pergu...,0.8,A resposta do modelo está embasada na base de ...,0.7,A base de conhecimento recuperada pelo algorit...,0.8,0.8,0.7,0.5
6,Como a cidade de Tóquio é retratada dentro de ...,Tóquio aparece quase como uma metrópole distóp...,A ambientação principal de Tokyo Ghoul é a pró...,A cidade de Tóquio em Tokyo Ghoul é retratada ...,['inhas entre \nbem e mal tornam-se tênues. \n...,0.8,A resposta do modelo está de acordo com a grou...,0.6,A resposta do modelo cobre alguns aspectos da ...,0.9,A resposta do modelo é muito relevante à pergu...,0.8,A resposta do modelo está embasada na base de ...,0.7,A base de conhecimento recuperada pelo algorit...,0.8,0.8,0.6,0.6
7,Qual o papel do café Anteiku na ambientação de...,"O café Anteiku, localizado em Setagaya Ward, s...",A ambientação principal de Tokyo Ghoul é a pró...,O café Anteiku representa um refúgio neutro pa...,"['olhos da maioria. \nO café Anteiku, localiza...",1.0,A resposta do modelo está de acordo com a grou...,0.8,A resposta do modelo cobre a maior parte da ba...,1.0,A re

In [26]:
file_name = "final_dataset"
df.to_excel(f"{file_name}.xlsx")

### Visualizando a correlação entre as métricas

Uma vez as métricas terem sido formadas, faz-se necessário verificar se elas possuem uma correlação, de modo a verificar o quão confiável ou relacionado está as métricas formadas pelo LLM as a Judge - em termos quantitativos -, considerando-a junto as métricas encontradas pelo BERT Score e pela cálculo de similaridade pelos cossenos. Isso é importante, pois elas mensuram a assertividade da resposta do modelo e da base de conhecimento recuperada a partir de uma distância vetorial, promovendo uma análise mais objetiva, quando comparada ao judge. 

Como estratégia de aferição da correlação entre as métricas, pode ser adotado por exemplo: 
- Matriz de correlação 
- Cálculo da esferecidade de Barlett 
- teste de hipótese

O cálculo de esferecidade de Barlett verifica se a matriz de correlação formada apresenta colinearidade entre os valores ou não, na forma em que se sim, ela difere significativamente de uma matriz identidade. O teste de hipótese varia de acordo com cada cenário. Contudo, pode-se adotar o seguinte pensamento (ignorando especificidades): se há uma quantidade de amostra menor que 30, adota-se o teste-t; se não, adota-se o teste-z. Com efeito, os testes serão aplicados no primeiro momento para a análise da qualidade da resposta do modelo e, posteriormente, para a base de conhecimento recuperada. 

**Ponto de atenção :**

Tais medidas de correlação estão sendo empregadas para avaliar as métricas formadas pelo judge em relação as demais outras. Não obstante, a mesma abordagem pode ser analisada para avaliar se tais métricas se correlacionam com a curadoria humana. Isso é importante, pois, caso se correlacione, tais métricas podem ser adotadas como estratégia de avaliação para um mesmo conjunto de dados, eximindo a curadoria humana de prover curadoria para novos dados que partilham a mesma distribuição, representando maior celeridade durante o processo de avaliação da qualidade de resposta dos modelos generativos. 

In [6]:
# Selecionando as colunas que serão analisadas pelas métricas de correlação. 

response_columns = ["corretude_rubrica", "bert_score_response", "cossine similarity response"]
kb_columns = ["embasamento_rubrica", "bert_score_kb", "cossine similarity kb"]

In [7]:
df_metrics_response = df[response_columns]
df_metrics_kbs = df[kb_columns]

In [8]:
cor_metric_response = df_metrics_response.corr()
cor_metric_kb = df_metrics_kbs.corr()

#### Analisando a resposta do modelo

**Matriz de Correlação**

In [10]:
fig = px.imshow(
    cor_metric_response,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    origin="lower",
    labels=dict(x="Métrica", y="Métrica", color="Correlação"),
    title="Matrix"
)
fig.update_layout(
    width=600, height=600,
    xaxis_side="top"
)

fig.show()

**Esfericidade de Barlett**

In [13]:
chi2, p_value = calculate_bartlett_sphericity(cor_metric_response)
print(f"Chi-square = {chi2:.2f}, p-value = {p_value:.3f}")
if p_value < 0.05:
    print("Rejeita H₀: há correlação significativa entre variáveis.")
else:
    print("Falha em rejeitar H₀: não há correlação suficiente.")

Chi-square = inf, p-value = 0.000
Rejeita H₀: há correlação significativa entre variáveis.


**Teste de Hipótese**

In [17]:
a = df["corretude_rubrica"].values
b = df["bert_score_response"].values
c = df["cossine similarity response"].values

In [18]:
t_stat, p_value = ttest_rel(a, b)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = 0.215, p = 0.8316
Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).


In [19]:
t_stat, p_value = ttest_rel(a, c)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = 5.168, p = 0.0000
Rejeitamos H₀: as médias diferem significativamente (α=0.05).


In [20]:
t_stat, p_value = ttest_rel(b, c)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = 6.896, p = 0.0000
Rejeitamos H₀: as médias diferem significativamente (α=0.05).


#### Analisando a base de conhecimento

**Matrix de Correlação**

In [21]:
fig = px.imshow(
    cor_metric_kb,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    origin="lower",
    labels=dict(x="Métrica", y="Métrica", color="Correlação"),
    title="Matrix"
)
fig.update_layout(
    width=600, height=600,
    xaxis_side="top"
)

fig.show()

**Esfericidade de Barlett**

In [22]:
chi2, p_value = calculate_bartlett_sphericity(cor_metric_response)
print(f"Chi-square = {chi2:.2f}, p-value = {p_value:.3f}")
if p_value < 0.05:
    print("Rejeita H₀: há correlação significativa entre variáveis.")
else:
    print("Falha em rejeitar H₀: não há correlação suficiente.")

Chi-square = inf, p-value = 0.000
Rejeita H₀: há correlação significativa entre variáveis.


**Teste de Hipótese**

In [23]:
a = df["embasamento_rubrica"].values
b = df["bert_score_kb"].values
c = df["cossine similarity kb"].values

In [24]:
t_stat, p_value = ttest_rel(a, b)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = -5.676, p = 0.0000
Rejeitamos H₀: as médias diferem significativamente (α=0.05).


In [25]:
t_stat, p_value = ttest_rel(a, c)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = 3.932, p = 0.0005
Rejeitamos H₀: as médias diferem significativamente (α=0.05).


In [26]:
t_stat, p_value = ttest_rel(c, b)

# 4. Exibir e interpretar
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Rejeitamos H₀: as médias diferem significativamente (α=0.05).")
else:
    print("Falha em rejeitar H₀: sem evidência de diferença nas médias (α=0.05).")

t = -12.069, p = 0.0000
Rejeitamos H₀: as médias diferem significativamente (α=0.05).


### Perfomance de resposta do modelo

**Considerando a resposta**

In [33]:
metrics_df = pd.DataFrame({
    "BERT Score-F1": df["bert_score_response"].values,
    "Cosine Similarity": df["cossine similarity response"].values,
    "Judge": df["corretude_rubrica"].values,
})
df_long = metrics_df.melt(var_name="Métrica", value_name="Score")

freq_df = (
    df_long
      .groupby(["Métrica", "Score"])
      .size()
      .reset_index(name="Frequência")
)

# Opcional: para ordenar os scores numericamente
freq_df = freq_df.sort_values(["Métrica", "Score"])

fig = px.bar(
    freq_df,
    x="Score",
    y="Frequência",
    color="Métrica",
    barmode="group",
    facet_col="Métrica",           # se quiser barras separadas por métrica
    category_orders={"Score": sorted(freq_df["Score"].unique())},
    title="Frequência de Scores por Métrica",
    labels={"Score":"Pontuação", "Frequência":"Nº de Ocorrências"}
)


fig.show()


**Considerando a base de conhecimento**

In [35]:
metrics_df = pd.DataFrame({
    "BERT Score-F1": df["bert_score_kb"].values,
    "Cosine Similarity": df["cossine similarity kb"].values,
    "Judge": df["embasamento_rubrica"].values,
})
df_long = metrics_df.melt(var_name="Métrica", value_name="Score")

freq_df_kb = (
    df_long
      .groupby(["Métrica", "Score"])
      .size()
      .reset_index(name="Frequência")
)

# Opcional: para ordenar os scores numericamente
freq_df_kb = freq_df_kb.sort_values(["Métrica", "Score"])

fig = px.bar(
    freq_df_kb,
    x="Score",
    y="Frequência",
    color="Métrica",
    barmode="group",
    facet_col="Métrica",           # se quiser barras separadas por métrica
    category_orders={"Score": sorted(freq_df_kb["Score"].unique())},
    title="Frequência de Scores por Métrica",
    labels={"Score":"Pontuação", "Frequência":"Nº de Ocorrências"}
)


fig.show()
